# Generation des grilles 87_0 sans et avec BelalpSolar

Ce notebook reconstruit la grille pandapower depuis le fichier Excel historique, ajoute BelalpSolar dans une copie de la grille, puis exporte les deux scenarios utilises par `FFOR.ipynb`.

In [ ]:
import copy
import math
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower as pp

data_dir = Path("Data")
data_dir.mkdir(exist_ok=True)
grid_file = data_dir / "87_0_grid.xlsx"
grid_id = "87_0"
nodes_file = data_dir / "nodes_valais.csv"
if not nodes_file.exists():
    raise FileNotFoundError("Impossible de trouver Data/nodes_valais.csv.")

with open(data_dir / "ffor_data.pkl", "rb") as f_pkl:
    ffor_data = pickle.load(f_pkl)

belalp_name = "BelalpSolar PV"
belalp_latitude = 46.39776
belalp_longitude = 7.97738
belalp_p_mw_nominal = 5.7
belalp_q_mvar = 0.0

## Grille 87_0 sans BelalpSolar

In [ ]:
def excel_value(row, key, default=None):
    value = row.get(key, default)
    return default if pd.isna(value) else value


def build_network_from_excel(path):
    """Reconstruit la grille sans utiliser pp.from_excel, incompatible avec certains anciens exports."""
    parameters = pd.read_excel(path, sheet_name="parameters").iloc[0]
    net = pp.create_empty_network(
        name="grid_87_0_without_belalp",
        f_hz=float(parameters["f_hz"]),
        sn_mva=float(parameters["sn_mva"]),
    )

    for _, row in pd.read_excel(path, sheet_name="bus").iterrows():
        pp.create_bus(
            net,
            index=int(row["Unnamed: 0"]),
            name=excel_value(row, "name"),
            vn_kv=float(row["vn_kv"]),
            type=excel_value(row, "type", "b"),
            zone=excel_value(row, "zone"),
            in_service=bool(row["in_service"]),
        )

    for _, row in pd.read_excel(path, sheet_name="line").iterrows():
        pp.create_line_from_parameters(
            net,
            index=int(row["Unnamed: 0"]),
            name=excel_value(row, "name"),
            from_bus=int(row["from_bus"]),
            to_bus=int(row["to_bus"]),
            length_km=float(row["length_km"]),
            r_ohm_per_km=float(row["r_ohm_per_km"]),
            x_ohm_per_km=float(row["x_ohm_per_km"]),
            c_nf_per_km=float(row["c_nf_per_km"]),
            g_us_per_km=float(excel_value(row, "g_us_per_km", 0.0)),
            max_i_ka=float(row["max_i_ka"]),
            df=float(excel_value(row, "df", 1.0)),
            parallel=int(excel_value(row, "parallel", 1)),
            type=excel_value(row, "type"),
            in_service=bool(row["in_service"]),
        )

    for _, row in pd.read_excel(path, sheet_name="trafo").iterrows():
        pp.create_transformer_from_parameters(
            net,
            index=int(row["Unnamed: 0"]),
            name=excel_value(row, "name"),
            hv_bus=int(row["hv_bus"]),
            lv_bus=int(row["lv_bus"]),
            sn_mva=float(row["sn_mva"]),
            vn_hv_kv=float(row["vn_hv_kv"]),
            vn_lv_kv=float(row["vn_lv_kv"]),
            vk_percent=float(row["vk_percent"]),
            vkr_percent=float(row["vkr_percent"]),
            pfe_kw=float(row["pfe_kw"]),
            i0_percent=float(row["i0_percent"]),
            shift_degree=float(excel_value(row, "shift_degree", 0.0)),
            tap_side=excel_value(row, "tap_side"),
            tap_neutral=excel_value(row, "tap_neutral", np.nan),
            tap_min=excel_value(row, "tap_min", np.nan),
            tap_max=excel_value(row, "tap_max", np.nan),
            tap_step_percent=excel_value(row, "tap_step_percent", np.nan),
            tap_step_degree=excel_value(row, "tap_step_degree", np.nan),
            tap_pos=excel_value(row, "tap_pos", np.nan),
            parallel=int(excel_value(row, "parallel", 1)),
            df=float(excel_value(row, "df", 1.0)),
            in_service=bool(row["in_service"]),
        )

    for _, row in pd.read_excel(path, sheet_name="ext_grid").iterrows():
        pp.create_ext_grid(
            net,
            index=int(row["Unnamed: 0"]),
            name=excel_value(row, "name"),
            bus=int(row["bus"]),
            vm_pu=float(excel_value(row, "vm_pu", 1.0)),
            va_degree=float(excel_value(row, "va_degree", 0.0)),
            slack_weight=float(excel_value(row, "slack_weight", 1.0)),
            in_service=bool(row["in_service"]),
        )

    for _, row in pd.read_excel(path, sheet_name="load").iterrows():
        pp.create_load(
            net,
            index=int(row["Unnamed: 0"]),
            name=excel_value(row, "name"),
            bus=int(row["bus"]),
            p_mw=float(row["p_mw"]),
            q_mvar=float(excel_value(row, "q_mvar", 0.0)),
            const_z_p_percent=float(excel_value(row, "const_z_percent", 0.0)),
            const_z_q_percent=float(excel_value(row, "const_z_percent", 0.0)),
            const_i_p_percent=float(excel_value(row, "const_i_percent", 0.0)),
            const_i_q_percent=float(excel_value(row, "const_i_percent", 0.0)),
            sn_mva=excel_value(row, "sn_mva", np.nan),
            scaling=float(excel_value(row, "scaling", 1.0)),
            type=excel_value(row, "type", "wye"),
            in_service=bool(row["in_service"]),
        )

    pp.runpp(net, calculate_voltage_angles=True, numba=False)
    return net


net_without_belalp = build_network_from_excel(grid_file)
print(f"Grille sans BelalpSolar: {len(net_without_belalp.bus)} bus, {len(net_without_belalp.line)} lignes")
print(f"Power flow converge: {net_without_belalp.converged}")
display(net_without_belalp.res_ext_grid[["p_mw", "q_mvar"]])

## Grille 87_0 avec BelalpSolar

In [ ]:
def id_to_str(value):
    try:
        return str(int(float(value)))
    except Exception:
        return str(value)


def node_id_from_bus_name(name):
    match = re.search(r"_node_(.+)$", str(name))
    return match.group(1) if match else None


def haversine_km(lon1, lat1, lon2, lat2):
    radius_km = 6371.0
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * radius_km * math.asin(math.sqrt(a))


nodes_geo = pd.read_csv(nodes_file, encoding="utf-8-sig")
nodes_grid = nodes_geo[nodes_geo["grid_id"].astype(str) == grid_id].dropna(subset=["osmid", "longitude", "latitude"]).copy()
nodes_grid["node_id"] = nodes_grid["osmid"].apply(id_to_str)
nodes_grid["longitude"] = pd.to_numeric(nodes_grid["longitude"], errors="coerce")
nodes_grid["latitude"] = pd.to_numeric(nodes_grid["latitude"], errors="coerce")
node_lookup = nodes_grid.drop_duplicates("node_id").set_index("node_id")[["longitude", "latitude"]]

bus_geo = net_without_belalp.bus.copy()
bus_geo["bus"] = bus_geo.index.astype(int)
bus_geo["node_id"] = bus_geo["name"].apply(node_id_from_bus_name)
bus_geo = bus_geo.join(node_lookup, on="node_id").dropna(subset=["longitude", "latitude"]).set_index("bus")
if bus_geo.empty:
    raise ValueError("Aucun bus de la grille 87_0 n'a pu etre associe a des coordonnees.")

distance2 = (bus_geo["longitude"] - belalp_longitude) ** 2 + (bus_geo["latitude"] - belalp_latitude) ** 2
nearest_bus = int(distance2.idxmin())
nearest_lon = float(bus_geo.loc[nearest_bus, "longitude"])
nearest_lat = float(bus_geo.loc[nearest_bus, "latitude"])
connection_length_km = haversine_km(nearest_lon, nearest_lat, belalp_longitude, belalp_latitude)

line_type = net_without_belalp.line.get("type", pd.Series("", index=net_without_belalp.line.index)).astype(str).str.lower()
std_type = net_without_belalp.line.get("std_type", pd.Series("", index=net_without_belalp.line.index)).astype(str).str.lower()
line_name = net_without_belalp.line.get("name", pd.Series("", index=net_without_belalp.line.index)).astype(str).str.lower()
ohl_mask = line_type.eq("ol") | std_type.str.contains("ohl|al1|st1|overhead", regex=True) | line_name.str.contains("ohl|al1|st1|overhead", regex=True)
ohl_candidates = net_without_belalp.line[ohl_mask]
if len(ohl_candidates):
    ohl_template = ohl_candidates.iloc[0]
else:
    ohl_template = pd.Series({
        "r_ohm_per_km": 0.5939,
        "x_ohm_per_km": 0.372,
        "c_nf_per_km": 9.5,
        "g_us_per_km": 0.0,
        "max_i_ka": 0.12,
        "df": 1.0,
        "parallel": 1,
    })

net_with_belalp = copy.deepcopy(net_without_belalp)
net_with_belalp.name = "grid_87_0_with_belalp"
belalp_bus = pp.create_bus(
    net_with_belalp,
    vn_kv=float(net_with_belalp.bus.loc[nearest_bus, "vn_kv"]),
    name="bus_MV_87_0_BelalpSolar",
    type="n",
)
belalp_sgen = pp.create_sgen(
    net_with_belalp,
    bus=belalp_bus,
    p_mw=belalp_p_mw_nominal,
    q_mvar=belalp_q_mvar,
    name=belalp_name,
    type="PV",
)
belalp_line = pp.create_line_from_parameters(
    net_with_belalp,
    from_bus=nearest_bus,
    to_bus=belalp_bus,
    length_km=max(connection_length_km, 1e-4),
    r_ohm_per_km=float(ohl_template["r_ohm_per_km"]),
    x_ohm_per_km=float(ohl_template["x_ohm_per_km"]),
    c_nf_per_km=float(ohl_template["c_nf_per_km"]),
    g_us_per_km=float(ohl_template.get("g_us_per_km", 0.0)),
    max_i_ka=float(ohl_template["max_i_ka"]),
    df=float(ohl_template.get("df", 1.0)),
    parallel=int(ohl_template.get("parallel", 1)),
    type="ol",
    name="OHL_87_0_to_BelalpSolar",
)
pp.runpp(net_with_belalp, calculate_voltage_angles=True, numba=False)

print(f"Grille avec BelalpSolar: {len(net_with_belalp.bus)} bus, {len(net_with_belalp.line)} lignes")
print(f"BelalpSolar: sgen {belalp_sgen}, bus {belalp_bus}, raccorde au bus {nearest_bus}")
print(f"Longueur approximative de la ligne OHL: {connection_length_km:.2f} km")
print(f"Power flow converge: {net_with_belalp.converged}")
display(net_with_belalp.res_ext_grid[["p_mw", "q_mvar"]])

## Export des variables pour FFOR.ipynb

In [ ]:
def build_line_data(net):
    active_lines = net.line[net.line["in_service"]].copy()
    line_data = pd.DataFrame({
        "from_bus": active_lines["from_bus"].astype(int),
        "to_bus": active_lines["to_bus"].astype(int),
        "length": active_lines["length_km"].astype(float),
        "r": active_lines["r_ohm_per_km"] * active_lines["length_km"],
        "x": active_lines["x_ohm_per_km"] * active_lines["length_km"],
        "c": active_lines["c_nf_per_km"] * active_lines["length_km"] * 1e-9,
        "I_max": active_lines["max_i_ka"].astype(float),
    })
    line_data["g"] = line_data["r"] / (line_data["r"] ** 2 + line_data["x"] ** 2)
    line_data["b"] = -line_data["x"] / (line_data["r"] ** 2 + line_data["x"] ** 2)
    line_data["b_sh"] = 2 * np.pi * float(net.f_hz) * line_data["c"]
    line_data["S_max"] = np.sqrt(3) * line_data["I_max"] * line_data["from_bus"].map(net.bus["vn_kv"])
    return line_data.reset_index(drop=True)


def build_jacobians(nodes, line_data):
    bus_position = {bus: index for index, bus in enumerate(nodes)}
    n_nodes = len(nodes)
    J_Ptheta = np.zeros((n_nodes, n_nodes))
    J_QU = np.zeros((n_nodes, n_nodes))
    J_PU = np.zeros((n_nodes, n_nodes))
    for _, line in line_data.iterrows():
        i = bus_position[int(line["from_bus"])]
        j = bus_position[int(line["to_bus"])]
        gij = float(line["g"])
        bij = float(line["b"])
        bsh = float(line["b_sh"])
        J_Ptheta[i, i] -= bij
        J_Ptheta[j, j] -= bij
        J_Ptheta[i, j] += bij
        J_Ptheta[j, i] += bij
        J_QU[i, i] -= 2 * bsh + bij
        J_QU[j, j] -= 2 * bsh + bij
        J_QU[i, j] += bij
        J_QU[j, i] += bij
        J_PU[i, i] += gij
        J_PU[j, j] += gij
        J_PU[i, j] -= gij
        J_PU[j, i] -= gij
    return J_Ptheta, J_QU, J_PU, -J_PU


def build_ffor_scenario(net, name, extra_pv_bus=None, metadata=None):
    nodes = list(map(int, net.bus.index))
    line_data = build_line_data(net)
    J_Ptheta, J_QU, J_PU, J_Qtheta = build_jacobians(nodes, line_data)
    active_loads = net.load[net.load["in_service"]]
    load_p = -active_loads.groupby("bus")["p_mw"].sum()
    load_q = -active_loads.groupby("bus")["q_mvar"].sum()
    P_load = {bus: float(load_p.get(bus, 0.0)) for bus in nodes}
    Q_load = {bus: float(load_q.get(bus, 0.0)) for bus in nodes}
    P_pv_max = {bus: 0.0 for bus in nodes}
    Q_pv_max = {bus: 0.0 for bus in nodes}
    P_hp_max = {bus: 0.0 for bus in nodes}
    for bus, value in ffor_data["P_pv_max"].items():
        if int(bus) in P_pv_max:
            P_pv_max[int(bus)] = float(value)
    for bus, value in ffor_data["Q_pv_max"].items():
        if int(bus) in Q_pv_max:
            Q_pv_max[int(bus)] = abs(float(value))
    for bus, value in ffor_data["P_hp_max"].items():
        if int(bus) in P_hp_max:
            P_hp_max[int(bus)] = float(value)
    if extra_pv_bus is not None:
        P_pv_max[int(extra_pv_bus)] = belalp_p_mw_nominal
        Q_pv_max[int(extra_pv_bus)] = belalp_p_mw_nominal * np.tan(np.arccos(float(ffor_data["cos_phi"])))
    return {
        "name": name,
        "n_nodes": len(nodes),
        "nodes": nodes,
        "pcc_bus": int(net.ext_grid.loc[net.ext_grid["in_service"], "bus"].iloc[0]),
        "P_base": float(net.res_ext_grid["p_mw"].iloc[0]),
        "Q_base": float(net.res_ext_grid["q_mvar"].iloc[0]),
        "P_ref": -net.res_bus.reindex(nodes)["p_mw"].fillna(0.0).to_numpy(),
        "Q_ref": -net.res_bus.reindex(nodes)["q_mvar"].fillna(0.0).to_numpy(),
        "P_load": P_load,
        "Q_load": Q_load,
        "P_pv_max": P_pv_max,
        "Q_pv_max": Q_pv_max,
        "P_hp_max": P_hp_max,
        "line_data": line_data,
        "J_Ptheta": J_Ptheta,
        "J_QU": J_QU,
        "J_PU": J_PU,
        "J_Qtheta": J_Qtheta,
        "cos_phi": float(ffor_data["cos_phi"]),
        "V": float(ffor_data["V"]),
        "delta_Umin": float(ffor_data["delta_Umin"]),
        "delta_Umax": float(ffor_data["delta_Umax"]),
        "metadata": metadata or {},
    }


scenarios = {
    "without_belalp": build_ffor_scenario(net_without_belalp, "87_0 sans BelalpSolar"),
    "with_belalp": build_ffor_scenario(
        net_with_belalp,
        "87_0 avec BelalpSolar",
        extra_pv_bus=belalp_bus,
        metadata={
            "belalp_bus": int(belalp_bus),
            "nearest_bus": int(nearest_bus),
            "belalp_line": int(belalp_line),
            "connection_length_km": float(connection_length_km),
            "belalp_p_mw_nominal": float(belalp_p_mw_nominal),
        },
    ),
}

scenario_file = data_dir / "ffor_grid_scenarios.pkl"
with open(scenario_file, "wb") as f_pkl:
    pickle.dump(scenarios, f_pkl)
pp.to_pickle(net_without_belalp, str(data_dir / "grid_87_0_without_belalp.p"))
pp.to_pickle(net_with_belalp, str(data_dir / "grid_87_0_with_belalp.p"))

for key, scenario in scenarios.items():
    print(f"{key}: {scenario['n_nodes']} bus, {len(scenario['line_data'])} lignes, P_base={scenario['P_base']:.3f} MW, Q_base={scenario['Q_base']:.3f} MVAr")
print(f"Scenarios FFOR sauvegardes dans {scenario_file}")

## Visualisation des deux grilles

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.colors import sample_colorscale

pio.renderers.default = "vscode"

plot_coordinates = bus_geo[["longitude", "latitude"]].copy()
plot_coordinates.loc[int(belalp_bus)] = [belalp_longitude, belalp_latitude]
max_loading_common = max(
    float(net_without_belalp.res_line["loading_percent"].max()),
    float(net_with_belalp.res_line["loading_percent"].max()),
    1.0,
)
max_load_common = max(
    float(net_without_belalp.load["p_mw"].max()),
    float(net_with_belalp.load["p_mw"].max()),
    1e-9,
)


def add_bus_markers(fig, items, name, color, symbol, sizes):
    if not items:
        return
    fig.add_trace(go.Scatter(
        x=[item["longitude"] for item in items],
        y=[item["latitude"] for item in items],
        mode="markers+text",
        text=[str(item["label"]) for item in items],
        textposition="top center",
        textfont=dict(size=9),
        marker=dict(symbol=symbol, size=sizes, color=color, line=dict(color="black", width=1)),
        customdata=[[item["bus"], item["load_mw"], item["vm_pu"]] for item in items],
        hovertemplate=(
            "bus %{customdata[0]}<br>"
            "load: %{customdata[1]:.4f} MW<br>"
            "voltage: %{customdata[2]:.4f} pu<extra></extra>"
        ),
        name=name,
    ))


def build_interactive_grid(net, title):
    fig = go.Figure()
    for line_index, line in net.line.iterrows():
        from_bus = int(line["from_bus"])
        to_bus = int(line["to_bus"])
        if from_bus not in plot_coordinates.index or to_bus not in plot_coordinates.index:
            continue
        x0, y0 = plot_coordinates.loc[from_bus, ["longitude", "latitude"]]
        x1, y1 = plot_coordinates.loc[to_bus, ["longitude", "latitude"]]
        loading = float(net.res_line.loc[line_index, "loading_percent"])
        p_from = float(net.res_line.loc[line_index, "p_from_mw"])
        q_from = float(net.res_line.loc[line_index, "q_from_mvar"])
        loading_ratio = min(max(loading / max_loading_common, 0.0), 1.0)
        is_belalp_line = str(line.get("name", "")) == "OHL_87_0_to_BelalpSolar"
        fig.add_trace(go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=sample_colorscale("Turbo", loading_ratio)[0],
                width=2.0 + 3.0 * loading_ratio,
                dash="dash" if is_belalp_line else "solid",
            ),
            customdata=[[line_index, from_bus, to_bus, loading, p_from, q_from]] * 2,
            hovertemplate=(
                "line %{customdata[0]}<br>"
                "%{customdata[1]} -> %{customdata[2]}<br>"
                "loading: %{customdata[3]:.2f}%<br>"
                "P_from: %{customdata[4]:.3f} MW<br>"
                "Q_from: %{customdata[5]:.3f} MVAr<extra></extra>"
            ),
            showlegend=False,
        ))

    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            color=[0],
            colorscale="Turbo",
            cmin=0,
            cmax=max_loading_common,
            showscale=True,
            colorbar=dict(title="Loading (%)", x=1.02, len=0.75),
        ),
        hoverinfo="skip",
        showlegend=False,
    ))

    active_loads = net.load[net.load["in_service"]]
    load_by_bus = active_loads.groupby("bus")["p_mw"].sum() if len(active_loads) else pd.Series(dtype=float)
    pcc_bus = int(net.ext_grid.loc[net.ext_grid["in_service"], "bus"].iloc[0])
    plain, loads, special = [], [], []
    for bus in net.bus.index:
        bus = int(bus)
        if bus not in plot_coordinates.index:
            continue
        longitude, latitude = plot_coordinates.loc[bus, ["longitude", "latitude"]]
        item = {
            "bus": bus,
            "label": "BelalpSolar" if bus == int(belalp_bus) else bus,
            "longitude": float(longitude),
            "latitude": float(latitude),
            "load_mw": float(load_by_bus.get(bus, 0.0)),
            "vm_pu": float(net.res_bus.loc[bus, "vm_pu"]),
        }
        if bus == pcc_bus or bus == int(belalp_bus):
            special.append(item)
        elif item["load_mw"] > 0:
            loads.append(item)
        else:
            plain.append(item)

    add_bus_markers(fig, plain, "Bus", "lightgray", "circle", [7] * len(plain))
    add_bus_markers(fig, loads, "Load", "crimson", "circle", [9 + 28 * item["load_mw"] / max_load_common for item in loads])
    pcc_items = [item for item in special if item["bus"] == pcc_bus]
    belalp_items = [item for item in special if item["bus"] == int(belalp_bus)]
    add_bus_markers(fig, pcc_items, "PCC", "gold", "star", [24] * len(pcc_items))
    add_bus_markers(fig, belalp_items, "BelalpSolar", "limegreen", "diamond", [22] * len(belalp_items))

    fig.update_layout(
        title=f"{title} | max loading: {float(net.res_line['loading_percent'].max()):.2f}%",
        width=1150,
        height=800,
        dragmode="zoom",
        hovermode="closest",
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        legend=dict(title="Elements", x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.88)"),
        margin=dict(l=55, r=135, t=70, b=55),
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig


fig_without_belalp = build_interactive_grid(net_without_belalp, "Grille 87_0 sans BelalpSolar")
fig_with_belalp = build_interactive_grid(net_with_belalp, "Grille 87_0 avec BelalpSolar")
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
fig_without_belalp.write_html(output_dir / "grid_87_0_without_belalp.html")
fig_with_belalp.write_html(output_dir / "grid_87_0_with_belalp.html")
fig_without_belalp.show()
fig_with_belalp.show()